# Memory of a single conversation: first Python, then checkpointer

In the previous lesson, the loop sent only the new question to the agent, so the model did not receive previous turns. Here, we will keep the history of **one conversation in the kernel memory**. We do not save it to a file. After a kernel restart, we start over.

First, we will manage a standard Python dictionary ourselves. Then, we will see how the agent's checkpointer takes over this same task. To call the model, we use the shared project factory `get_model()`.


In [ ]:
# Only imports and minimal project path setup.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from model_config import get_model


In [ ]:
model = get_model()


## 1. Manually: the dictionary grows after each turn

`conversation` is a standard dictionary. In `messages`, we store the system message as well as subsequent questions and answers. Before each call, **my** passes the entire history up to that point along with the new question to the model. The model itself does not remember previous calls.


In [ ]:
conversation = {
    "messages": [
        {"role": "system", "content": "Answer briefly in English."},
    ]
}
print(conversation)


In [ ]:
def ask_manually(question: str) -> None:
    user_message = {"role": "user", "content": question}
    messages_for_model = conversation["messages"] + [user_message]
    answer = model.invoke(messages_for_model).content

    conversation["messages"].append(user_message)
    conversation["messages"].append({"role": "assistant", "content": answer})

    print("Answer:", answer)
    print("Number of messages in the dictionary:", len(conversation["messages"]))
    print(conversation)


In [ ]:
ask_manually("My name is Ola. Remember my name.")


In [ ]:
ask_manually("What is my name?")


After the first turn, the dictionary contains 3 messages: system, user, assistant. After the second turn, it contains 5. The second call received all 3 previous messages, which is why it can answer the question about the name. You can call `ask_manually(...)` again to add another turn.

## 2. LangChain: checkpointer stores history

`create_agent` with `InMemorySaver` appends the history automatically. In each call, we provide only the **new** question. A constant `thread_id` identifies this single conversation; there is no selecting other conversations or saving to disk here.


In [ ]:
checkpointer = InMemorySaver()
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="Answer briefly in English.",
    checkpointer=checkpointer,
)
config = {"configurable": {"thread_id": "one-conversation"}}


In [ ]:
first = agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Ola. Remember my name."}]},
    config=config,
)
print("Response:", first["messages"][-1].content)
print("Number of messages in the checkpoint:", len(first["messages"]))


In [ ]:
second = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config,
)
print("Answer:", second["messages"][-1].content)
print("Number of messages in the checkpoint:", len(second["messages"]))


In both variants, the next question is sent to the model with the previous context. The difference: in the first variant, we update the dictionary and pass the entire history ourselves; in the second, the checkpointer does this for the agent. `InMemorySaver` works only in the current process. The next lesson adds saving, reading, and changing the conversation ID.
